# AccessAI — Web app YOLO26x para Google Colab

Este cuaderno inicia una interfaz web temporal para analizar imágenes con el checkpoint local `yolo26x.pt`.

## Antes de ejecutar

1. En Google Colab, selecciona un entorno de ejecución con GPU.
2. Ejecuta las celdas en orden.
3. Cuando se solicite, carga el archivo `yolo26x.pt` desde tu equipo.
4. Abre el enlace de Gradio que aparece en la última celda.

El checkpoint es genérico: la app muestra las clases que contiene ese checkpoint y no aplica el mapeo experimental de cuatro categorías. Sus detecciones no han sido validadas como clases específicas de accesibilidad ni prueban que un lugar sea accesible o inaccesible.

El enlace temporal de Gradio es público mientras la app está activa. No cargues imágenes sensibles.


In [ ]:
%pip -q install ultralytics gradio

## 1. Comprobar PyTorch y GPU

El cuaderno se detiene si PyTorch no detecta una GPU; no cambia a CPU.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No se detectó una GPU. En Colab, cambia Tipo de entorno de ejecución a GPU y vuelve a ejecutar."
    )

DEVICE = 0
print("GPU:", torch.cuda.get_device_name(DEVICE))
print("CUDA de PyTorch:", torch.version.cuda)


## 2. Cargar el checkpoint YOLO26x

Si `yolo26x.pt` no está en el entorno de Colab, la celda abrirá el selector de archivos.

In [ ]:
from pathlib import Path
from google.colab import files
from ultralytics import YOLO

MODEL_PATH = Path.cwd() / "yolo26x.pt"

if not MODEL_PATH.is_file():
    print("Selecciona el archivo yolo26x.pt.")
    uploaded = files.upload()
    if not MODEL_PATH.is_file() and "yolo26x.pt" not in uploaded:
        raise FileNotFoundError(
            "No se recibió yolo26x.pt. Vuelve a ejecutar la celda y selecciona ese archivo."
        )

if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"No se encuentra el checkpoint: {MODEL_PATH}")

model = YOLO(str(MODEL_PATH))
print("Checkpoint cargado:", MODEL_PATH.name)
print("Clases leídas del checkpoint:")
for class_id, class_name in model.names.items():
    print(f"{class_id}: {class_name}")


## 3. Función de inferencia

La confianza inicial `0.25` y el tamaño `640` conservan los valores usados en el prototipo. La inferencia se fuerza a la GPU seleccionada.

In [ ]:
from PIL import Image


def analizar_imagen(image, confidence):
    if image is None:
        return None, [], "Carga una imagen para iniciar el análisis."

    image_rgb = image.convert("RGB")
    results = model.predict(
        source=image_rgb,
        conf=float(confidence),
        imgsz=640,
        device=DEVICE,
        verbose=False,
    )
    result = results[0]

    # Ultralytics devuelve plot() en orden BGR; PIL espera RGB.
    annotated_bgr = result.plot()
    annotated_rgb = Image.fromarray(annotated_bgr[:, :, ::-1].copy())

    rows = []
    boxes = result.boxes
    if boxes is not None:
        for index in range(len(boxes)):
            class_id = int(boxes.cls[index].item())
            score = float(boxes.conf[index].item())
            x1, y1, x2, y2 = [
                round(float(value), 1)
                for value in boxes.xyxy[index].tolist()
            ]

            if isinstance(result.names, dict):
                class_name = result.names.get(class_id, str(class_id))
            else:
                class_name = result.names[class_id]

            rows.append([
                class_name,
                round(score, 4),
                x1,
                y1,
                x2,
                y2,
            ])

    if rows:
        message = f"Detecciones: {len(rows)}. Confianza mínima: {float(confidence):.2f}."
    else:
        message = (
            f"No se detectaron objetos con confianza ≥ {float(confidence):.2f}. "
            "Esto no es una evaluación de accesibilidad."
        )

    return annotated_rgb, rows, message


## 4. Interfaz web

Sube una imagen, ajusta la confianza y pulsa **Analizar imagen**. La salida incluye la imagen anotada y una tabla con clase, confianza y coordenadas de cada caja.

In [ ]:
import gradio as gr

with gr.Blocks(title="AccessAI — YOLO26x") as demo:
    gr.Markdown(
        """# AccessAI · Detector YOLO26x

Checkpoint genérico; las etiquetas proceden de sus clases internas.
Las detecciones no son un dictamen de accesibilidad."""
    )

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(type="pil", label="Imagen de entrada")
            confidence = gr.Slider(
                minimum=0.0,
                maximum=1.0,
                value=0.25,
                step=0.01,
                label="Confianza mínima",
            )
            analyze_button = gr.Button("Analizar imagen", variant="primary")
        with gr.Column():
            output_image = gr.Image(type="pil", label="Imagen anotada")

    detections_table = gr.Dataframe(
        headers=["Clase", "Confianza", "x1", "y1", "x2", "y2"],
        datatype=["str", "number", "number", "number", "number", "number"],
        interactive=False,
        label="Detecciones",
    )
    status = gr.Markdown()

    analyze_button.click(
        fn=analizar_imagen,
        inputs=[input_image, confidence],
        outputs=[output_image, detections_table, status],
    )

demo.launch(share=True, show_error=True)
